# Train deepfake-detect on Kaggle

Trains the linear head on **FaceForensics++ (c23)** using the official train/val/test splits, then tests on **Celeb-DF v2**, a dataset the model never sees during training. That second number is the honest one.

**Before running**
1. Settings → Accelerator → **GPU T4 x2** (or P100). Turn **Internet on**.
2. Both datasets need an access request to their authors ([FaceForensics++](https://github.com/ondyari/FaceForensics), [Celeb-DF](https://github.com/yuezunli/celeb-deepfakeforensics)). Their licences forbid redistribution, so upload your copy as a **private** Kaggle dataset and attach it with *Add Input*.
3. Set the two paths in the next code cell to wherever your upload lands under `/kaggle/input/`.

Expected runtime: ~1–2 h, mostly video decoding and face detection. Embeddings are cached, so re-runs take minutes.

In [ ]:
REPO = "https://github.com/CodeRafay/deepfake-detect"
BRANCH = "main"
!git clone -q -b $BRANCH $REPO /kaggle/working/deepfake-detect
!pip install -q /kaggle/working/deepfake-detect pytest
!python -m pytest -q /kaggle/working/deepfake-detect/tests

In [ ]:
import json, urllib.request
from pathlib import Path

FFPP = Path("/kaggle/input/YOUR-FFPP-UPLOAD")        # contains original_sequences/ and manipulated_sequences/
CELEBDF = Path("/kaggle/input/YOUR-CELEBDF-UPLOAD")  # contains Celeb-real/, Celeb-synthesis/, YouTube-real/, List_of_testing_videos.txt
METHODS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
DATA, CDF = Path("/kaggle/working/data"), Path("/kaggle/working/celebdf")

def link(src, dst):
    if src.exists() and not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        dst.symlink_to(src)

# Official FF++ splits are by source-video pair, so no identity appears in two splits.
for split in ["train", "val", "test"]:
    url = f"https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/splits/{split}.json"
    for a, b in json.load(urllib.request.urlopen(url)):
        for v in (a, b):
            link(FFPP / f"original_sequences/youtube/c23/videos/{v}.mp4", DATA / split / "real" / f"{v}.mp4")
        for m in METHODS:
            for x, y in ((a, b), (b, a)):
                link(FFPP / f"manipulated_sequences/{m}/c23/videos/{x}_{y}.mp4", DATA / split / "fake" / f"{m}_{x}_{y}.mp4")

# Celeb-DF official test list: "1 path" = real, "0 path" = fake.
for line in (CELEBDF / "List_of_testing_videos.txt").read_text().splitlines():
    if line.strip():
        lab, rel = line.split()
        link(CELEBDF / rel, CDF / ("real" if lab == "1" else "fake") / rel.replace("/", "_"))

for d in [DATA / s / c for s in ["train", "val", "test"] for c in ["real", "fake"]] + [CDF / "real", CDF / "fake"]:
    print(f"{d}: {len(list(d.glob('*')))} videos")

In [ ]:
!cd /kaggle/working && python -m deepfake_detect.train --data data --eval celebdf=celebdf --out head.safetensors

Paste the table above into the README's **Results** section. Report it as is, even if the Celeb-DF number is unflattering. Next, sanity-check a few test videos:

In [ ]:
from deepfake_detect import Detector
det = Detector("/kaggle/working/head.safetensors")
for p in sorted((CDF / "real").glob("*"))[:3] + sorted((CDF / "fake").glob("*"))[:3]:
    r = det.predict(p)
    print(p.parent.name, r["label"], r["score"])

## Publish the head (optional)
Add a Kaggle secret `HF_TOKEN` (write access), then upload. Users get it automatically with `Detector()` once `DEFAULT_HEAD` in `model.py` points to your repo.

In [ ]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
repo = "CodeRafay/deepfake-detect"
api.create_repo(repo, exist_ok=True)
for f in ["head.safetensors", "head.metrics.json"]:
    api.upload_file(path_or_fileobj=f"/kaggle/working/{f}", path_in_repo=f, repo_id=repo)